ShopBondhu AWS Deployment Commands Log

1. Local Machine: Key Permissions & SSH Agent Setup
```bash
chmod 400 shopbondhu_key.pem

# Start SSH agent and add key for forwarding to private backend
eval "$(ssh-agent -s)"
ssh-add shopbondhu_key.pem
```
we can do the same by cpoying the ssh_key to public instance but it is volnurable to have a sensitive key in the public instance.

2. Web Server: Connect, Install Nginx, and Start Service

```Bash
# Connect to the public web server
ssh -A ubuntu@3.71.5.35

# Package upgrade and Nginx installation
sudo apt update -y
sudo apt install nginx -y

# Enable and start Nginx daemon
sudo systemctl enable nginx
sudo systemctl start nginx
sudo systemctl status nginx
```
![nginx_status](./screenshots/13_nginx_up_and_running_from_public_ec2.png)

3. Local Machine: Transfer Frontend Landing Assets

```Bash
scp -i shopbondhu-key.pem -r ./frontend/* ubuntu@3.71.5.35:~/landing/
```
we can use git clone from our public repo as well. both works fine. I have proof.

4. Web Server: Deploy Frontend to Nginx Web Root

```bash
# Replace default Nginx landing files
sudo rm -rf /var/www/html/*
sudo cp -r ~/landing/* /var/www/html/
sudo chown -R www-data:www-data /var/www/html
sudo systemctl reload nginx
```
![website_ruunig](./screenshots/14_we-server%20running%20in%20public%20ec2.png)

5. Jumped to Backend Server & Test Outbound NAT Gateway

```bash
# From Web Server terminal, SSH to backend using private IP
ssh ubuntu@10.0.1.7
# Verify internet access through NAT Gateway
sudo apt update
curl -I [https://google.com](https://google.com)

# Verify instance has no public routing interface
ubuntu@ip-10-0-1-7:~$ ip route show
default via 10.0.1.1 dev ens5 proto dhcp src 10.0.1.7 metric 100
10.0.0.2 via 10.0.1.1 dev ens5 proto dhcp src 10.0.1.7 metric 100
10.0.1.0/24 dev ens5 proto kernel scope link src 10.0.1.7 metric 100
10.0.1.1 dev ens5 proto dhcp scope link src 10.0.1.7 metric 100

6. Local Machine: Verify Backend Isolation (Failed)


```Bash
ping -c 3 10.0.1.7
ssh -i shopbondhu-key.pem ubuntu@10.0.1.7
```